# 第 05 章：Agent 中间件 (Agent Middleware)

根据讲义，本章你将掌握以下核心能力：
- **`with_listeners` 事件监听**：实现对执行过程的审计记录。
- **`with_fallbacks` 容错切换**：构建即便主模型宕机也依然稳固的架构。
- **`Custom Redactor` 隐私打码**：编写自己的拦截器，保障内容安全。
- **`HITL` 流程管控**：实现高风险操作的人工最后确认。

## 1. 统一模型声明与环境准备
我们将定义两个模型：主模型 (DeepSeek) 和备用模型 (OpenAI)。

In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage, HumanMessage

load_dotenv()

# 主模型 (Primary)
primary_llm = ChatOpenAI(
    model="gpt-4o-mini", 
    api_key=os.getenv("OPENAI_API_KEY")
)
# 备用模型 (Backup)
backup_llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://api.deepseek.com"
)

print("双模型架构就绪。")

双模型架构就绪。


## 2. 独立演示：with_listeners (审计黑匣子)
监听器会独立记录任务的启动 (`on_start`) 和结束 (`on_end`)。

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1. 定义组件
prompt = ChatPromptTemplate.from_template("你是一个助手，请回答：{question}")
output_parser = StrOutputParser() # 👈 规范的关键：专门的输出解析器

# 2. 准备带有监听器的 LLM
def audit_metadata(run):
    usage = run.outputs.get("llm_output", {}).get("token_usage", {})
    print(f"[Log] ID: {run.id} | 消耗 Tokens: {usage.get('total_tokens', 0)}")

monitored_llm = backup_llm.with_listeners(on_end=audit_metadata)

# 3. 规范的链式组合 (使用 | 符号)
# 数据流：Dict -> Prompt -> LLM -> String
chain = prompt | monitored_llm | output_parser

# 4. 调用
result = chain.invoke({"question": "你是谁？"})

print(f"最终业务逻辑拿到的结果: {result}")

[Log] ID: 019d5289-b563-7933-aec2-02cb5717323c | 消耗 Tokens: 161
最终业务逻辑拿到的结果: 你好！我是DeepSeek，由深度求索公司创造的AI助手。😊

我是一个纯文本模型，擅长回答各种问题、协助写作、分析问题、编程帮助等等。虽然我不支持多模态识别，但我可以处理你上传的文件（如图像、txt、pdf、ppt、word、excel等），从中读取文字信息来帮助你。

我的特点是：
- 完全免费使用
- 拥有128K的上下文长度
- 支持联网搜索（需要手动开启）
- 可以通过官方应用商店下载App使用

我的知识截止到2024年7月，会以热情细腻的方式为你提供帮助。有什么问题或需要协助的地方，尽管告诉我吧！我会尽力帮助你。✨


## 3. 独立演示：with_fallbacks (备用发电机)
模拟主模型宕机。我们将故意传入一个错误的 API Key 给主模型。

In [17]:
# 一个必定失效的模型
broken_llm = ChatOpenAI(model="deepseek-chat", api_key="expired_token")

# 建立容错链
# 故意放主链路都是故障的
reliable_llm = broken_llm.with_fallbacks([primary_llm])

print("--- 启动混沌实验 (Chaos Test) ---")
try:
    res = reliable_llm.invoke("帮我数一下单词 Apple 里有几个字母。")
    print(f"\n最终响应成功！来自备用架构: {res.content}")
except Exception as e:
    print(f"容错失败: {e}")

--- 启动混沌实验 (Chaos Test) ---
容错失败: Error code: 401 - {'error': {'message': 'Incorrect API key provided: expired_*oken. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


## 4. 独立演示：Custom Redactor (隐私打码器)
通过正则拦截管道中的敏感手机号。

In [18]:
import re

def mask_pii(text: str) -> str:
    return re.sub(r"(\d{3})\d{4}(\d{4})", r"\1****\2", text)

# 包装为中间件节点
redact_node = RunnableLambda(lambda msg: mask_pii(msg.content))

print("--- 测试脱敏拦截 ---")
test_input = AIMessage(content="好的，转账给 13812345678 的操作已记录。")
print(f"拦截前: {test_input.content}")
print(f"拦截后: {redact_node.invoke(test_input)}")

--- 测试脱敏拦截 ---
拦截前: 好的，转账给 13812345678 的操作已记录。
拦截后: 好的，转账给 138****5678 的操作已记录。


## 5. 本章终极 Lab：金融转账智能助手 (组装)
我们将上述所有独立模块，加上**人工审批干预**，组装成一个最终的 Agent。

In [19]:
from langchain_core.tools import tool

@tool
def execute_transfer(amount: float, target_phone: str):
    """执行极其敏感的转账操作。"""
    return f"[银行系统] 对 {target_phone} 的 {amount} 元转账成功。流水号: TXN_777"

def human_guard(intent):
    """中间件：人工最后确认层"""
    print(f"\n[👮 安全审查] AI 申请转账 {intent['amount']} 元给 {intent['target_phone']}")
    if input("请输入 Y 确认执行, N 拦截: ").upper() == "Y":
        return True
    return False

print("--- 启动全流程 Lab 模拟 ---")
# 场景：AI 决定转账 (模拟 LLM 意图)
mock_intent = {"amount": 500.0, "target_phone": "13988889999"}

if human_guard(mock_intent):
    raw_res = execute_transfer.invoke(mock_intent)
    # 经过脱敏后输出给用户
    print(f"\n用户视界（已脱敏）: {mask_pii(raw_res)}")
else:
    print("\n⛔ 操作已被拦截，已记录进审计日志。")

--- 启动全流程 Lab 模拟 ---

[👮 安全审查] AI 申请转账 500.0 元给 13988889999

⛔ 操作已被拦截，已记录进审计日志。
